In [1]:
import pandas as pd
pd.set_option('display.max_columns', 20)
import numpy as np
from sklearn.model_selection import KFold
from sklearn.naive_bayes import MultinomialNB

In [2]:
df = pd.read_excel('credit_scoring_dqlab.xlsx')
print(df.head())

  kode_kontrak  pendapatan_setahun_juta kpr_aktif  durasi_pinjaman_bulan  \
0   AGR-000001                      295        YA                     48   
1   AGR-000011                      271        YA                     36   
2   AGR-000030                      159     TIDAK                     12   
3   AGR-000043                      210        YA                     12   
4   AGR-000049                      165     TIDAK                     36   

   jumlah_tanggungan rata_rata_overdue  risk_rating  
0                  5      61 - 90 days            4  
1                  5      61 - 90 days            4  
2                  0       0 - 30 days            1  
3                  3      46 - 60 days            3  
4                  0      31 - 45 days            2  


In [3]:
df.drop('kode_kontrak', axis=1, inplace=True)
print(df.head())

   pendapatan_setahun_juta kpr_aktif  durasi_pinjaman_bulan  \
0                      295        YA                     48   
1                      271        YA                     36   
2                      159     TIDAK                     12   
3                      210        YA                     12   
4                      165     TIDAK                     36   

   jumlah_tanggungan rata_rata_overdue  risk_rating  
0                  5      61 - 90 days            4  
1                  5      61 - 90 days            4  
2                  0       0 - 30 days            1  
3                  3      46 - 60 days            3  
4                  0      31 - 45 days            2  


In [4]:
print('Rasio kemunculan label:')
print(pd.concat([df['risk_rating'].value_counts(), 100*df['risk_rating'].value_counts()/len(df)], axis=1))

Rasio kemunculan label:
             count      count
risk_rating                  
3              291  32.333333
1              227  25.222222
2              159  17.666667
4              120  13.333333
5              103  11.444444


In [5]:
y= df.pop('risk_rating').to_list()
y = [4 if  label == 5 else label for label in y ]
y = np.array(y)
print('\n Dataset:')
print(df.head())



 Dataset:
   pendapatan_setahun_juta kpr_aktif  durasi_pinjaman_bulan  \
0                      295        YA                     48   
1                      271        YA                     36   
2                      159     TIDAK                     12   
3                      210        YA                     12   
4                      165     TIDAK                     36   

   jumlah_tanggungan rata_rata_overdue  
0                  5      61 - 90 days  
1                  5      61 - 90 days  
2                  0       0 - 30 days  
3                  3      46 - 60 days  
4                  0      31 - 45 days  


In [6]:
def convert_kpr_aktif (kpr_aktif):
    if kpr_aktif == 'YA':
        return 1
    return 0

In [7]:
df['kpr_aktive'] = df['kpr_aktif'].apply(convert_kpr_aktif)
print(df.head())

   pendapatan_setahun_juta kpr_aktif  durasi_pinjaman_bulan  \
0                      295        YA                     48   
1                      271        YA                     36   
2                      159     TIDAK                     12   
3                      210        YA                     12   
4                      165     TIDAK                     36   

   jumlah_tanggungan rata_rata_overdue  kpr_aktive  
0                  5      61 - 90 days           1  
1                  5      61 - 90 days           1  
2                  0       0 - 30 days           0  
3                  3      46 - 60 days           1  
4                  0      31 - 45 days           0  


In [8]:
print('Rasio kemunculan setiap kategori rata-rata_overdue:')
print(pd.concat([df['rata_rata_overdue'].value_counts(), 100*df['rata_rata_overdue'].value_counts(normalize=True).rename('percentasi rata-rata-overdue')], axis=1))


Rasio kemunculan setiap kategori rata-rata_overdue:
                   count  percentasi rata-rata-overdue
rata_rata_overdue                                     
46 - 60 days         291                     32.333333
0 - 30 days          227                     25.222222
31 - 45 days         159                     17.666667
61 - 90 days         120                     13.333333
> 90 days            103                     11.444444


In [9]:
def change_overdue(overdue):
    if overdue == '0 - 30 days':
        return 0
    elif overdue == '31 - 45 days':
        return 1
    elif overdue == '46 - 60 days':
        return 2
    elif overdue == '61 - 90 days':
        return 3
    else:
        return 4
df['rata_rata_overdue'] = df['rata_rata_overdue'].apply(change_overdue)
print('\n Lima baris dataset')
print(df.head())


 Lima baris dataset
   pendapatan_setahun_juta kpr_aktif  durasi_pinjaman_bulan  \
0                      295        YA                     48   
1                      271        YA                     36   
2                      159     TIDAK                     12   
3                      210        YA                     12   
4                      165     TIDAK                     36   

   jumlah_tanggungan  rata_rata_overdue  kpr_aktive  
0                  5                  3           1  
1                  5                  3           1  
2                  0                  0           0  
3                  3                  2           1  
4                  0                  1           0  


In [10]:
#Dataset untuk Feature Matrix sebagai X
X = df.to_numpy()
print('dimensi dari variabel:',X.shape)

dimensi dari variabel: (900, 6)


In [11]:
#menginisialisasi object KFold dengan jumlah kelompok data = 5 nilai random_state kita gunakan reproducibility (agar data acak yang kita dapatkan untuk setiap kelompok data selalu sama)
kf = KFold(n_splits=5, shuffle=True, random_state=57)

for i, (train_index, test_index) in enumerate(kf.split(X)):
    X_train, X_test = X[train_index], y[train_index]
    y_train, y_test = X[test_index], y[test_index]

    print('percobaan ke:', i+1)
    print("10 indeks data latih pertama:", train_index[:10])
    print("10 indeks data testing pertama:", test_index[:10])
    print("============================================================")

percobaan ke: 1
10 indeks data latih pertama: [ 0  1  2  3  4  5  6  7  8 13]
10 indeks data testing pertama: [ 9 10 11 12 15 25 29 32 44 52]
percobaan ke: 2
10 indeks data latih pertama: [ 0  2  4  5  6  8  9 10 11 12]
10 indeks data testing pertama: [ 1  3  7 14 20 28 37 43 49 59]
percobaan ke: 3
10 indeks data latih pertama: [ 0  1  2  3  4  5  6  7  9 10]
10 indeks data testing pertama: [ 8 13 22 23 27 30 31 33 38 39]
percobaan ke: 4
10 indeks data latih pertama: [ 1  3  4  5  6  7  8  9 10 11]
10 indeks data testing pertama: [ 0  2 16 17 19 21 24 35 36 42]
percobaan ke: 5
10 indeks data latih pertama: [ 0  1  2  3  7  8  9 10 11 12]
10 indeks data testing pertama: [ 4  5  6 18 26 34 40 47 50 54]


In [12]:
#berdasarkan hasil di atas menunjukkan kabhwa jumlah percobaab  akan diilakukan sebanyak nilai n yang dispesifikasikan. 
# berdasarkan 10 index data dan data testing pertama pada setiap percobaab, index yg telah muncul sebagai index pada 
# testing. selain itu data latih dan data testing yang digunakan pada setiap percobbaan selalu memiliki perbedaan.

In [13]:
scores_test = []
scores_train = []
 
#meminta object kf untuk memecah data X ke sejumlah n kelompok dan mengiterasi setiap train_index dan test_index
for i, (train_index, test_index) in enumerate(kf.split(X)):
    X_train, y_train = X[train_index], y[train_index]
    X_test, y_test = X[test_index], y[test_index]
    model = MultinomialNB()
    #menspesifikasikan data latih beserta labelnya untuk dipelajari oleh model
    model.fit(X_train, y_train)
    print(f'Selesai melatih data dengan strategi validasi 5-Fold ke-{i+1}.')

ValueError: could not convert string to float: 'YA'